# Silver Pipeline - Subscriptions

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col, row_number
from pyspark.sql.window import Window
from delta.tables import DeltaTable
from functools import reduce

print("[INFO] Subscriptions Silver pipeline started")

In [ ]:
spark  # Databricks provided SparkSession
batch_id = dbutils.widgets.get("batch_id") if 'dbutils' in globals() else None

## 1. Read Bronze subscriptions

In [ ]:
bronze_path = "/Volumes/datalake_catalog/datalake_schema/bronze/subscriptions"

df_bronze = (
    spark.read.format("delta").load(bronze_path)
    .drop("source_identifier", "batch_id", "id", "ingest_time")
)

df_bronze.show(5)

## 2. Deduplication (latest record per subscription)

In [ ]:
w = Window.partitionBy("subscription_id", "created_at").orderBy(col("created_at").desc())

df1 = (
    df_bronze
    .withColumn("rn", row_number().over(w))
    .filter(col("rn") == 1)
    .drop("rn")
)

df1_quarantine = df_bronze.subtract(df1)

## 3. NULL validation

In [ ]:
df2 = df1.filter(
    col("subscription_id").isNotNull() &
    col("user_id").isNotNull() &
    col("plan_id").isNotNull() &
    col("start_date").isNotNull() &
    col("status").isNotNull()
)

df2_quarantine = df1.subtract(df2)

## 4. Type casting

In [ ]:
df3 = df2.select(
    col("subscription_id").cast("bigint"),
    col("user_id").cast("bigint"),
    col("plan_id").cast("bigint"),
    col("start_date").cast("date"),
    col("end_date").cast("date"),
    F.lower(F.trim(col("status"))).alias("status"),
    col("current_mrr").cast("decimal(10,2)"),
    col("created_at").cast("timestamp")
)

## 5. Status validation

In [ ]:
valid_status = ["active", "cancelled", "expired", "trial"]

df4 = df3.filter(col("status").isin(valid_status))
df4_quarantine = df3.filter(~col("status").isin(valid_status))

## 6. Reference validation (users)

In [ ]:
users_silver_path = "/Volumes/datalake_catalog/datalake_schema/silver/users"

df_users = spark.read.format("delta").load(users_silver_path)

df_valid_users = df_users.select("user_id").distinct()

df5 = df4.join(df_valid_users, "user_id", "left_semi")
df5_quarantine = df4.join(df_valid_users, "user_id", "left_anti")

## 7. Date validation

In [ ]:
df6 = df5.filter(
    col("end_date").isNull() | (col("start_date") <= col("end_date"))
)

df6_quarantine = df5.filter(
    col("end_date").isNotNull() & (col("start_date") > col("end_date"))
)

## 8. MRR business rule validation

In [ ]:
df7 = df6.filter(
    ((col("status") == "trial") & (col("current_mrr") == 0)) |
    ((col("status").isin("active", "cancelled", "expired")) & (col("current_mrr") > 0))
)

df7_quarantine = df6.subtract(df7)

## 9. Churn feature

In [ ]:
df8 = df7.withColumn(
    "churn",
    F.when(
        col("status").isin("cancelled", "expired") |
        (col("end_date").isNotNull() & (col("end_date") <= F.current_date())),
        "Yes"
    ).otherwise("No")
)

## 10. Combine quarantines

In [ ]:
quarantine_dfs = [
    df1_quarantine,
    df2_quarantine,
    df4_quarantine,
    df5_quarantine,
    df6_quarantine,
    df7_quarantine
]

df_quarantine_all = reduce(
    lambda a, b: a.unionByName(b, allowMissingColumns=True),
    quarantine_dfs
)

print("[INFO] Quarantine rows:", df_quarantine_all.count())

## 11. Write to Silver Delta

In [ ]:
silver_path = "/Volumes/datalake_catalog/datalake_schema/silver/subscriptions"

df_final = df8

if DeltaTable.isDeltaTable(spark, silver_path):
    delta = DeltaTable.forPath(spark, silver_path)
    
    delta.alias("t").merge(
        df_final.alias("s"),
        "t.subscription_id = s.subscription_id"
    ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()
else:
    df_final.write.format("delta").mode("overwrite").save(silver_path)

print("[SUCCESS] Written to Silver subscriptions")